In [ ]:
%pip uninstall sagemaker -y
%pip install "sagemaker>=2.0,<3.0" -q

#### imports

In [8]:
import sagemaker
from sagemaker.feature_store.feature_group import FeatureGroup
import boto3

In [9]:
import awswrangler as wr

df_meta = wr.athena.read_sql_query(
    "SELECT * FROM pneumonia_db.image_metadata",
    database="pneumonia_db"
)

# Remove checkpoint files if any
df_meta = df_meta[~df_meta["s3_key"].str.contains(".ipynb_checkpoint")]

print(f"Total images to process: {len(df_meta)}")

2026-06-02 15:28:00,430	WARNING services.py:2137 -- WARNING: The object store is using /tmp instead of /dev/shm because /dev/shm has only 891269120 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=1.89gb' to 'docker run' (or add it to the run_options list in a Ray cluster config). Make sure to set this to more than 30% of available RAM.


2026-06-02 15:28:01,628	INFO worker.py:2007 -- Started a local Ray instance.


/opt/conda/lib/python3.12/site-packages/ray/_private/worker.py:2046: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Total images to process: 33757


In [11]:
# This will show you the S3 LOCATION the table is registered with
table_info = wr.catalog.get_table_types(
    database="pneumonia_db",
    table="image_metadata"
)
print(table_info)

{'image_id': 'string', 's3_key': 'string', 'file_name': 'string', 'split': 'string', 'label': 'string', 'file_type': 'string', 'source': 'string', 'file_size': 'bigint'}


In [12]:
details = wr.catalog.get_table_parameters(
    database="pneumonia_db",
    table="image_metadata"
)
print(details)

{'skip.header.line.count': '1', 'EXTERNAL': 'TRUE', 'transient_lastDdlTime': '1780414036'}


In [13]:
# Best way to see the location
import boto3
glue = boto3.client("glue")
response = glue.get_table(DatabaseName="pneumonia_db", Name="image_metadata")
print(response["Table"]["StorageDescriptor"]["Location"])

s3://pneumonia-data-set-group-4/pneumonia-project/metadata


In [6]:



sess = sagemaker.Session()

feature_group = FeatureGroup(
    name="pneumonia_training_manifest_1780368800",
    sagemaker_session=sess
)

desc = feature_group.describe()
print(desc["FeatureGroupStatus"])

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:8                                                                                    │
│                                                                                                  │
│    5 │   sagemaker_session=sess                                                                  │
│    6 )                                                                                           │
│    7                                                                                             │
│ ❱  8 desc = feature_group.describe()                                                             │
│    9 print(desc["FeatureGroupStatus"])                                                           │
│   10                                                                                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/telemetry/telemetry_logging.py:171 in wrapper  │
│                                                                                                  │
│   168 │   │   │   │   │   caught_ex = e                                                          │
│   169 │   │   │   │   finally:                                                                   │
│   170 │   │   │   │   │   if caught_ex:                                                          │
│ ❱ 171 │   │   │   │   │   │   raise caught_ex                                                    │
│   172 │   │   │   │   │   return response  # pylint: disable=W0150                               │
│   173 │   │   │   else:                                                                          │
│   174 │   │   │   │   logger.debug(                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/telemetry/telemetry_logging.py:142 in wrapper  │
│                                                                                                  │
│   139 │   │   │   │   start_timer = perf_counter()                                               │
│   140 │   │   │   │   try:                                                                       │
│   141 │   │   │   │   │   # Call the original function                                           │
│ ❱ 142 │   │   │   │   │   response = func(*args, **kwargs)                                       │
│   143 │   │   │   │   │   stop_timer = perf_counter()                                            │
│   144 │   │   │   │   │   elapsed = stop_timer - start_timer                                     │
│   145 │   │   │   │   │   extra += f"&x-latency={round(elapsed, 2)}"                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/feature_store/feature_group.py:809 in describe │
│                                                                                                  │
│    806 │   │   Returns:                                                                          │
│    807 │   │   │   Response dict from the service.                                               │
│    808 │   │   """                                                                               │
│ ❱  809 │   │   return self.sagemaker_session.describe_feature_group(                             │
│    810 │   │   │   feature_group_name=self.name, next_token=next_token                           │
│    811 │   │   )                                                                                 │
│    812                                                                                           │
│                                                            